# Mashtots — классификация армянских рукописных букв (78 классов)

Ноутбук для запуска прямо в Kaggle: `Save & Run All (Commit)` обучает CNN и
кладёт `submission.csv` в `/kaggle/working`. Интернет не нужен, всё есть в
образе Kaggle.

Подходит и к [Mashtots Dataset](https://www.kaggle.com/competitions/mashtots-dataset),
и к [Mashtots Dataset v2](https://www.kaggle.com/competitions/mashtots-dataset-v2):
каталог с классами и тестовая часть ищутся автоматически в `/kaggle/input`.
Первая ячейка печатает инвентаризацию входа — если автопоиск что-то не нашёл,
по её выводу сразу видно, какой путь подставить.

**Перед запуском:** справа `Add Input` → соревнование, и `Settings` →
`Accelerator: GPU`. На T4 полный прогон занимает считанные минуты.

## Что важно в этой модели

Наивная реализация этой задачи очень легко застревает на `loss = ln 78 = 4.3567`
и `accuracy = 1/78 = 0.0128`, то есть выдаёт равномерное распределение. Две
типичные причины, обе учтены здесь:

1. **Ненормализованный вход.** `cv2.imread` даёт `uint8 0..255`; без нормализации
   предактивации огромные, и первый же шаг Adam уводит почти все ReLU в
   отрицательную область. Здесь `Rescaling(1/255)` стоит **внутри** модели, так
   что нормализация применяется и при инференсе автоматически.
2. **Узкое место в голове.** Слой вида `Dense(2, relu)` перед `Dense(78)`
   физически не даёт обучиться: градиент через два подряд сужающихся ReLU
   вырождается. Голова здесь без сужений.

Отдельная, не такая известная ловушка: `BatchNormalization` с дефолтным
`momentum=0.99`. Скользящие статистики, которыми BN пользуется на инференсе,
обновляются как `m ← 0.99·m + (1-0.99)·батч` и уходят от инициализации только за
тысячи шагов. Пока они не сошлись, train-метрики выглядят прекрасно, а
val-метрики стоят на `1/78` при растущем val-loss. Поэтому ниже `momentum=0.9`.

## 1. Импорты, конфигурация, GPU

In [ ]:
import os
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
IMG_SIZE = 64            # родное разрешение датасета, апскейл не нужен
EPOCHS = 40              # реально отработает меньше: есть early stopping
VAL_FRACTION = 0.10
TEST_FRACTION = 0.10     # локальная честная оценка, в обучении не участвует
MAX_PER_CLASS = None     # поставьте 50 для быстрой проверки пайплайна
USE_TTA = True           # усреднение предсказаний по сдвигам на ±2 пикселя
REFIT_ON_ALL = False     # см. раздел 7

INPUT_DIR = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path(".")
MODEL_PATH = WORK_DIR / "mashtots_cnn.keras"
SUBMISSION_PATH = WORK_DIR / "submission.csv"

keras.utils.set_random_seed(SEED)
rng = np.random.default_rng(SEED)
sns.set_theme(style="whitegrid")

GPUS = tf.config.list_physical_devices("GPU")
for gpu in GPUS:
    tf.config.experimental.set_memory_growth(gpu, True)
if GPUS:
    # float16 для вычислений, float32 для весов: примерно вдвое быстрее на T4/P100
    keras.mixed_precision.set_global_policy("mixed_float16")
BATCH_SIZE = 256 if GPUS else 64

print(f"tensorflow {tf.__version__} | keras {keras.__version__}")
print(f"GPU: {[g.name for g in GPUS] or 'нет (будет CPU, медленно)'}")
print(f"политика точности: {keras.mixed_precision.global_policy().name} | batch: {BATCH_SIZE}")
print(f"результаты пишем в: {WORK_DIR}")

## 2. Что лежит во входных данных

Стандартный первый шаг в Kaggle: посмотреть на смонтированные данные. Для CSV
печатаются колонки и первые строки — по `sample_submission.csv` сразу видно
требуемый формат ответа.

In [ ]:
def show_input_tree(base: Path = INPUT_DIR, max_items: int = 12, max_kids: int = 3) -> None:
    if not base.is_dir():
        print(f"{base} не найден — ноутбук запущен вне Kaggle")
        return

    for comp in sorted(p for p in base.iterdir() if p.is_dir()):
        print(f"\n=== {comp}")
        items = sorted(comp.iterdir())
        for item in items[:max_items]:
            if item.is_dir():
                kids = sorted(item.iterdir())
                n_dirs = sum(k.is_dir() for k in kids)
                print(f"  {item.name}/ — записей {len(kids)}, из них папок {n_dirs}")
                for k in kids[:max_kids]:
                    print(f"      {k.name}{'/' if k.is_dir() else ''}")
                if len(kids) > max_kids:
                    print("      ...")
            else:
                print(f"  {item.name} — {item.stat().st_size / 2**20:.2f} MiB")
        if len(items) > max_items:
            print(f"  ... ещё {len(items) - max_items}")

        for csv_path in sorted(comp.glob("*.csv")):
            head = pd.read_csv(csv_path, nrows=3)
            shown = list(head.columns[:8])
            tail = f" ... (всего {len(head.columns)})" if len(head.columns) > 8 else ""
            print(f"\n  {csv_path.name}: колонки {shown}{tail}")
            display(head.iloc[:, :8])


show_input_tree()

## 3. Загрузка обучающих данных

Каталог классов ищется обходом `/kaggle/input` — годятся и `Train/`, и
`Train/Train/`, и любая другая вложенность. Читаем в `uint8`: 70 060 изображений
64×64 — это 287 МиБ, а во `float32` было бы 1.1 ГиБ. Нормализацию делает слой
`Rescaling` внутри модели.

Файлы и классы перебираются отсортированными (воспроизводимость), посторонние
расширения отсекаются, нечитаемые файлы пропускаются, а не роняют цикл.

In [ ]:
IMAGE_EXT = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".pgm"}


def list_class_dirs(root: Path) -> list[Path]:
    return sorted((p for p in root.iterdir() if p.is_dir() and p.name.isdigit()),
                  key=lambda p: int(p.name))


def list_images(directory: Path) -> list[Path]:
    return sorted(p for p in directory.rglob("*")
                  if p.is_file() and p.suffix.lower() in IMAGE_EXT)


def find_class_root(min_classes: int = 10, max_depth: int = 4) -> Path:
    """Каталог, в котором лежит >= min_classes папок с числовыми именами."""
    bases = sorted(p for p in INPUT_DIR.iterdir() if p.is_dir()) if INPUT_DIR.is_dir() else []
    bases += [Path("."), Path("data/mashtots")]

    for base in bases:
        if not base.is_dir():
            continue
        for dirpath, dirnames, _ in os.walk(base):
            if len([d for d in dirnames if d.isdigit()]) >= min_classes:
                return Path(dirpath)
            if len(Path(dirpath).relative_to(base).parts) >= max_depth:
                dirnames.clear()
            else:
                # в папки-классы спускаться незачем: там только картинки
                dirnames[:] = [d for d in dirnames if not d.isdigit()]

    raise FileNotFoundError(
        "не найден каталог с папками-классами. Проверьте, что соревнование "
        "добавлено через Add Input, и сверьтесь с выводом ячейки выше."
    )


def load_dataset(root: Path, img_size: int = IMG_SIZE, max_per_class: int | None = None):
    class_dirs = list_class_dirs(root)
    samples = []
    for cdir in class_dirs:
        samples += [(f, int(cdir.name)) for f in list_images(cdir)[:max_per_class]]

    X = np.empty((len(samples), img_size, img_size, 1), dtype=np.uint8)
    y = np.empty(len(samples), dtype=np.int16)
    step = max(1, len(samples) // 5)

    n, resized, skipped = 0, 0, 0
    for i, (path, label) in enumerate(samples):
        img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        if img is None:
            skipped += 1
            continue
        if img.shape != (img_size, img_size):
            img = cv2.resize(img, (img_size, img_size), interpolation=cv2.INTER_AREA)
            resized += 1
        X[n, :, :, 0] = img
        y[n] = label
        n += 1
        if (i + 1) % step == 0:
            print(f"  прочитано {i + 1}/{len(samples)}")

    print(f"классов: {len(class_dirs)} | загружено: {n} | ресайз: {resized} | пропущено: {skipped}")
    return X[:n], y[:n]


CLASS_ROOT = find_class_root()
print("каталог классов:", CLASS_ROOT)

X, y = load_dataset(CLASS_ROOT, max_per_class=MAX_PER_CLASS)
NUM_CLASSES = int(y.max()) + 1
print(f"X: {X.shape} {X.dtype} ({X.nbytes / 2**20:.1f} MiB) | классов: {NUM_CLASSES}")

In [ ]:
counts = pd.Series(y).value_counts().sort_index()
ratio = counts.max() / counts.min()
print(f"на класс: min={counts.min()}, median={int(counts.median())}, max={counts.max()}")
print(f"дисбаланс max/min = {ratio:.2f} -> "
      f"{'веса классов не нужны' if ratio < 1.5 else 'стоит рассмотреть class_weight'}")

fig, axes = plt.subplots(1, 2, figsize=(14, 3.2))
axes[0].bar(counts.index, counts.values, width=1.0)
axes[0].set(title="Изображений по классам", xlabel="класс")
axes[1].hist(X[:: max(1, len(X) // 500)].ravel(), bins=50)
axes[1].set(title="Значения пикселей", xlabel="значение", yscale="log")
plt.tight_layout()
plt.show()

idx = rng.choice(len(X), size=min(16, len(X)), replace=False)
fig, axes = plt.subplots(2, 8, figsize=(13, 3.6))
for ax, i in zip(axes.ravel(), idx):
    ax.imshow(X[i, :, :, 0], cmap="gray")
    ax.set_title(f"class {y[i]}", fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Разбиение train / val / test

Стратифицированно, 80 / 10 / 10. `val` нужен для early stopping и планировщика
LR, `test` не участвует ни в обучении, ни в выборе модели — это единственная
честная локальная оценка. Судить по `val` нельзя: по нему выбираются веса.

In [ ]:
X_train, X_hold, y_train, y_hold = train_test_split(
    X, y, test_size=VAL_FRACTION + TEST_FRACTION, random_state=SEED, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_hold, y_hold,
    test_size=TEST_FRACTION / (VAL_FRACTION + TEST_FRACTION),
    random_state=SEED, stratify=y_hold
)
del X_hold, y_hold

for name, a, b in [("train", X_train, y_train), ("val", X_val, y_val), ("test", X_test, y_test)]:
    print(f"{name:<6} {len(a):>7} | классов {len(np.unique(b))}")

## 5. Модель

* `Rescaling` и аугментация — **слоями внутри модели**. `Rescaling` работает и на
  инференсе, поэтому расхождение препроцессинга между обучением и предсказанием
  невозможно; `Random*`-слои Keras сами отключаются вне обучения.
* **Без горизонтального отражения**: буквы зеркально несимметричны, флип
  превратил бы часть классов в мусор. Только поворот / сдвиг / зум, а пустота
  после трансформации заливается нулями — фон в датасете чёрный, и дефолтный
  `fill_mode="reflect"` затащил бы в кадр куски штриха.
* Три блока `Conv-BN-ReLU ×2 → MaxPool → Dropout`: пара свёрток 3×3 даёт
  рецептивное поле 5×5 дешевле, чем одна 5×5.
* `BatchNormalization(momentum=0.9)` — см. вступление.
* Голова без сужений: `Flatten(8·8·128) → Dense(256) → BN → Dropout → Dense(78)`,
  всего около 2.5 М параметров.
* Последний слой явно `dtype="float32"`: под `mixed_float16` softmax в половинной
  точности теряет устойчивость.

In [ ]:
BN_MOMENTUM = 0.9


def build_model(img_size: int = IMG_SIZE, num_classes: int = NUM_CLASSES) -> keras.Model:
    inputs = keras.Input(shape=(img_size, img_size, 1), name="image")

    x = layers.Rescaling(1.0 / 255)(inputs)
    x = layers.RandomRotation(0.06, fill_mode="constant", fill_value=0.0)(x)
    x = layers.RandomTranslation(0.08, 0.08, fill_mode="constant", fill_value=0.0)(x)
    x = layers.RandomZoom(0.10, fill_mode="constant", fill_value=0.0)(x)

    for filters in (32, 64, 128):
        for _ in range(2):
            x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
            x = layers.BatchNormalization(momentum=BN_MOMENTUM)(x)
            x = layers.Activation("relu")(x)
        x = layers.MaxPooling2D(2)(x)
        x = layers.Dropout(0.25)(x)

    x = layers.Flatten()(x)
    x = layers.Dense(256, use_bias=False)(x)
    x = layers.BatchNormalization(momentum=BN_MOMENTUM)(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(0.40)(x)
    outputs = layers.Dense(num_classes, activation="softmax", dtype="float32", name="probs")(x)

    return keras.Model(inputs, outputs, name="mashtots_cnn")


def compile_model(m: keras.Model) -> keras.Model:
    m.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy", keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3")],
    )
    return m


model = compile_model(build_model())
model.summary()

## 6. Обучение

`verbose=2` — одна строка на эпоху: прогресс-бары раздувают лог закоммиченного
ноутбука на десятки мегабайт.

Как читать лог, если что-то пошло не так: если **и** train, **и** val стоят на
`1/78` — проблема в данных или архитектуре. Если train растёт, а val стоит на
`1/78` при растущем val-loss — виноваты слои, по-разному работающие в train и
inference, то есть почти всегда BatchNorm с несошедшимися статистиками.

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=8, restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5, verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        MODEL_PATH, monitor="val_accuracy", save_best_only=True, verbose=0
    ),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=2,
)

In [ ]:
hist = pd.DataFrame(history.history)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
hist[["loss", "val_loss"]].plot(ax=axes[0], title="Loss")
axes[0].axhline(np.log(NUM_CLASSES), ls="--", c="r", label=f"ln({NUM_CLASSES}) — уровень «не учится»")
hist[["accuracy", "val_accuracy"]].plot(ax=axes[1], title="Accuracy")
axes[1].axhline(1 / NUM_CLASSES, ls="--", c="r", label=f"1/{NUM_CLASSES} — случайное угадывание")
for ax in axes:
    ax.set_xlabel("эпоха")
    ax.legend()
plt.tight_layout()
plt.show()

print(f"лучшая эпоха по val_accuracy: {int(np.argmax(history.history['val_accuracy'])) + 1}"
      f" из {len(hist)}")

## 7. Честная оценка на отложенном тесте

In [ ]:
for name, value in model.evaluate(X_test, y_test, batch_size=BATCH_SIZE,
                                  verbose=0, return_dict=True).items():
    print(f"test {name:<10} {value:.4f}")

probs = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)
y_pred = probs.argmax(axis=1)

report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
per_class = (pd.DataFrame(report).T.loc[lambda d: d.index.str.isdigit()]
             .astype({"support": int}).sort_values("f1-score"))
print("\n10 самых трудных классов (по f1):")
print(per_class.head(10).round(3))

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=np.arange(NUM_CLASSES))

plt.figure(figsize=(10, 8))
sns.heatmap(cm, cmap="viridis", square=True, cbar_kws={"shrink": 0.7})
plt.title(f"Confusion matrix ({NUM_CLASSES} классов)")
plt.xlabel("предсказано")
plt.ylabel("истина")
plt.tight_layout()
plt.show()

off = cm.copy()
np.fill_diagonal(off, 0)
pairs = [(a, b) for a, b in np.dstack(
    np.unravel_index(np.argsort(off, axis=None)[::-1], off.shape))[0][:10] if off[a, b] > 0]
if not pairs:
    print("перепутанных пар нет")
else:
    print("Чаще всего путаются (истина -> предсказание, случаев):")
    for a, b in pairs:
        print(f"  {a:>2} -> {b:>2} : {off[a, b]}")

## 8. Необязательно: переобучение на всех данных

Для лидерборда обычно выгодно доучиться на 100 % данных: `val` и `test` — это
ещё 20 % изображений. Число эпох берём то, на котором был максимум `val_accuracy`
в основном прогоне, — иначе на полных данных нечем остановиться.

Цена: локальная оценка из раздела 7 перестаёт относиться к отправляемой модели.
Поэтому по умолчанию выключено; поставьте `REFIT_ON_ALL = True` в разделе 1,
когда качество основного прогона вас устроит.

In [ ]:
if REFIT_ON_ALL:
    best_epochs = int(np.argmax(history.history["val_accuracy"])) + 1
    print(f"переобучение на всех {len(X)} изображениях, {best_epochs} эпох")
    keras.utils.set_random_seed(SEED)
    model = compile_model(build_model())
    model.fit(X, y, epochs=best_epochs, batch_size=BATCH_SIZE, verbose=2)
    model.save(MODEL_PATH)
else:
    print("REFIT_ON_ALL = False — для submission берём модель из раздела 6")

## 9. Submission

Тест соревнования бывает в двух видах, поддерживаются оба: каталог картинок
(`new_test/`) и таблица с развёрнутыми пикселями (`new_test.csv`). Читается тем
же кодом, что и обучающая часть.

Два момента, на которых обычно теряют баллы:

* **Порядок и формат id.** Предсказания раскладываются по `id` из
  `sample_submission.csv`, а не по порядку файлов; `123` и `123.png` считаются
  одним и тем же id. Ячейка печатает, сколько id удалось сопоставить.
* **TTA.** Усредняем предсказания по сдвигам на ±2 пикселя (`np.roll`,
  детерминированно). Отражения не используем — буквы несимметричны.

In [ ]:
def stack_images(paths) -> tuple[np.ndarray, list[Path]]:
    """Батч из файлов + список фактически прочитанных путей (битые пропускаем)."""
    out = np.empty((len(paths), IMG_SIZE, IMG_SIZE, 1), dtype=np.uint8)
    kept = []
    for i, path in enumerate(paths):
        img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue
        if img.shape != (IMG_SIZE, IMG_SIZE):
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
        out[len(kept), :, :, 0] = img
        kept.append(paths[i])
    return out[:len(kept)], kept


def find_competition_test(base: Path, prefer: Path | None = None):
    """(images, ids, sample_submission_path) или None, если тест не найден.

    prefer — каталог соревнования, из которого взята обучающая часть. Если
    подключено сразу несколько соревнований, он просматривается первым, иначе
    тест и train можно молча взять из разных источников.
    """
    comps = sorted(p for p in base.iterdir() if p.is_dir()) if base.is_dir() else [Path(".")]
    if prefer is not None:
        comps = [prefer] + [c for c in comps if c != prefer]

    for comp in comps:
        sample = next(iter(sorted(comp.rglob("sample_submission.csv"))), None)

        for name in ("new_test", "Test", "test"):
            for d in sorted(comp.rglob(name)):
                if not d.is_dir():
                    continue
                files = list_images(d)
                if files:
                    imgs, kept = stack_images(files)
                    return imgs, [p.name for p in kept], sample

        for csv_path in sorted(comp.rglob("new_test.csv")) + sorted(comp.rglob("test.csv")):
            df = pd.read_csv(csv_path)
            num = df.select_dtypes(include="number")
            pixel_cols = [c for c in num.columns if num[c].between(0, 255).all()]
            if len(pixel_cols) >= IMG_SIZE * IMG_SIZE:
                pix = num[pixel_cols[-IMG_SIZE * IMG_SIZE:]].to_numpy(dtype=np.uint8)
                imgs = pix.reshape(-1, IMG_SIZE, IMG_SIZE, 1)
                ids = (df[df.columns[0]].astype(str).tolist()
                       if len(pixel_cols) < len(df.columns) else list(range(len(df))))
                return imgs, ids, sample
    return None


def predict_with_tta(m: keras.Model, images: np.ndarray, use_tta: bool = True) -> np.ndarray:
    shifts = [(0, 0), (0, 2), (0, -2), (2, 0), (-2, 0)] if use_tta else [(0, 0)]
    total = np.zeros((len(images), NUM_CLASSES), dtype=np.float32)
    for dy, dx in shifts:
        batch = images if (dy, dx) == (0, 0) else np.roll(images, (dy, dx), axis=(1, 2))
        total += m.predict(batch, batch_size=BATCH_SIZE, verbose=0)
    return total / len(shifts)


COMPETITION_DIR = next(
    (p for p in [CLASS_ROOT, *CLASS_ROOT.parents] if p.parent == INPUT_DIR), None
)
test_data = find_competition_test(INPUT_DIR, prefer=COMPETITION_DIR)

if test_data is None:
    print("тест соревнования не найден — submission не создан.\n"
          "Проверьте Add Input и вывод ячейки с инвентаризацией.")
else:
    test_images, test_ids, sample_path = test_data
    print(f"тестовых изображений: {len(test_images)} | sample_submission: {sample_path}")

    test_probs = predict_with_tta(model, test_images, USE_TTA)
    test_pred = test_probs.argmax(axis=1)
    print(f"TTA: {'вкл' if USE_TTA else 'выкл'} | средняя уверенность: {test_probs.max(axis=1).mean():.3f}")

In [ ]:
def build_submission(ids, preds, sample_path: Path | None) -> pd.DataFrame:
    """Раскладываем предсказания по id из sample_submission, а не по порядку файлов."""
    by_id = {}
    for raw_id, cls in zip(ids, preds):
        key = str(raw_id)
        by_id[key] = int(cls)
        by_id[Path(key).stem] = int(cls)     # '123.png' и '123' — один и тот же id

    if sample_path is None or not Path(sample_path).is_file():
        print("sample_submission.csv не найден — пишем в порядке файлов, колонки Id/Category")
        return pd.DataFrame({"Id": [str(i) for i in ids], "Category": [int(p) for p in preds]})

    sample = pd.read_csv(sample_path)
    id_col = sample.columns[0]
    target_col = sample.columns[1] if len(sample.columns) > 1 else "Category"

    keys = sample[id_col].astype(str)
    matched = keys.map(lambda k: by_id.get(k, by_id.get(Path(k).stem)))
    missing = int(matched.isna().sum())

    if missing == len(sample):
        print("!! ни один id из sample_submission не совпал с именами файлов.\n"
              "   Пишем в порядке файлов — сверьте формат id вручную перед отправкой.")
        return pd.DataFrame({id_col: [str(i) for i in ids], target_col: [int(p) for p in preds]})

    if missing:
        print(f"!! нет предсказаний для {missing} из {len(sample)} id — заполнены нулём")
    print(f"сопоставлено {len(sample) - missing} из {len(sample)} id из sample_submission")

    out = sample.copy()
    out[target_col] = matched.fillna(0).astype(int)
    return out[[id_col, target_col]]


if test_data is not None:
    submission = build_submission(test_ids, test_pred, sample_path)
    submission.to_csv(SUBMISSION_PATH, index=False)
    print(f"\n{SUBMISSION_PATH}: {submission.shape[0]} строк, колонки {list(submission.columns)}")
    display(submission.head())

## 10. Проверка файла перед отправкой

Дешевле поймать несоответствие здесь, чем получить отказ от лидерборда.

In [ ]:
if not SUBMISSION_PATH.is_file():
    print("submission.csv не создан — отправлять нечего")
else:
    sub = pd.read_csv(SUBMISSION_PATH)
    id_col, target_col = sub.columns[0], sub.columns[1]
    checks = {
        "строк": len(sub),
        "id уникальны": bool(sub[id_col].is_unique),
        "пропусков нет": bool(sub.notna().all().all()),
        "классы в диапазоне": bool(sub[target_col].between(0, NUM_CLASSES - 1).all()),
        "разных классов": int(sub[target_col].nunique()),
    }
    if test_data is not None and sample_path is not None and Path(sample_path).is_file():
        sample = pd.read_csv(sample_path)
        checks["совпадает с sample по строкам"] = len(sub) == len(sample)
        checks["совпадает с sample по id"] = bool(
            set(sub[id_col].astype(str)) == set(sample[sample.columns[0]].astype(str))
        )
    for k, v in checks.items():
        print(f"  {k}: {v}")

    plt.figure(figsize=(12, 3))
    plt.bar(*np.unique(sub[target_col], return_counts=True), width=1.0)
    plt.title("Распределение предсказанных классов")
    plt.xlabel("класс")
    plt.tight_layout()
    plt.show()

## 11. Куда двигаться дальше

1. **`REFIT_ON_ALL = True`** — плюс 20 % обучающих данных, обычно самый дешёвый прирост.
2. **Ансамбль** 3–5 прогонов с разными `SEED` и усреднением вероятностей.
3. **Резидуальные блоки** вместо простых `Conv-BN-ReLU`: на 70 тыс. изображений
   более глубокая сеть уже окупается.
4. **Косинусный планировщик LR** (`keras.optimizers.schedules.CosineDecay`) вместо
   `ReduceLROnPlateau`.
5. **Label smoothing** 0.05 через `CategoricalCrossentropy` — в рукописном тексте
   есть объективно неоднозначные образцы, и смягчение метки снижает переуверенность.
6. **Разбор confusion matrix**: пары «заглавная/строчная» одной буквы путаются чаще
   всего; для них может помочь двухступенчатая схема (сначала буква, потом регистр).